# 08 — Comparison with Black-Box Uplift Models

This notebook compares simple uplift learners against stronger non-linear models.

The goal is not to pretend that simple models always win. The goal is to discuss the trade-off between performance, stability, and explainability.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np

from uplift_marketing.data import generate_synthetic_uplift_data

from sklearn.model_selection import train_test_split

from uplift_marketing.features import infer_feature_spec, split_features_treatment_outcome
from uplift_marketing.models import TLearner, make_base_classifier, make_black_box_classifier
from uplift_marketing.evaluation import qini_curve, area_under_qini


data = generate_synthetic_uplift_data(n_rows=5_000, random_state=42)
spec = infer_feature_spec(data, extra_excluded=["customer_id", "visit"])
x, treatment, y = split_features_treatment_outcome(data, spec)

x_train, x_test, t_train, t_test, y_train, y_test = train_test_split(
    x, treatment, y, test_size=0.3, random_state=42, stratify=treatment.astype(str) + "_" + y.astype(str)
)

models = {
    "logistic_t_learner": TLearner(spec=spec, base_model=make_base_classifier(42), random_state=42),
    "random_forest_t_learner": TLearner(spec=spec, base_model=make_black_box_classifier(42), random_state=42),
}

rows = []
for name, model in models.items():
    model.fit(x_train, t_train, y_train)
    uplift = model.predict_uplift(x_test)
    curve = qini_curve(y_test.to_numpy(), t_test.to_numpy(), uplift)
    rows.append({"model": name, "area_under_qini": area_under_qini(curve)})

pd.DataFrame(rows)


## Final discussion

A black-box uplift model may improve ranking performance.

But campaign teams still need to understand the policy, monitor harms, preserve holdout groups, and validate incrementality through experimentation.
